Packages Download

In [1]:
!uv pip install --upgrade openai openai-agents pypdf python-dotenv requests gradio

Resolved 79 packages in 3.45s
 Downloaded hf-xet
Prepared 7 packages in 285ms
Uninstalled 7 packages in 4.47s
Installed 7 packages in 5.25s
 - hf-xet==1.7.1
 + hf-xet==1.8.0
 - huggingface-hub==2.3.0
 + huggingface-hub==2.4.0
 - openai==3.40.0
 + openai==3.41.0
 - opentelemetry-api==1.45.1
 + opentelemetry-api==1.45.2
 - pydantic==2.14.1
 + pydantic==2.14.2
 - pydantic-core==2.50.1
 + pydantic-core==2.50.2
 - pytz==2026.5
 + pytz==2026.6


In [2]:
from dotenv import load_dotenv
import os

from openai import AsyncAzureOpenAI
from agents import (
    Agent,
    Runner,
    function_tool,
    OpenAIChatCompletionsModel,
    set_tracing_disabled,
    SQLiteSession,
    handoff,
)
from agents.extensions.handoff_prompt import RECOMMENDED_PROMPT_PREFIX
from pypdf import PdfReader
import gradio as gr

c:\Users\Danie\Downloads\Final Project\FinalProject_IT3025C\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


API Key Checkover.

In [3]:
load_dotenv(override=True)

AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
DEPLOYMENT = "gpt-5.4-ptu"

print("Endpoint loaded:", bool(AZURE_OPENAI_ENDPOINT))
print("API key loaded:", bool(AZURE_OPENAI_API_KEY))

Endpoint loaded: True
API key loaded: True


Disabling Tracer.

In [4]:
set_tracing_disabled(True)

azure_client = AsyncAzureOpenAI(
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
    api_version="2024-10-21",
)

azure_model = OpenAIChatCompletionsModel(
    model=DEPLOYMENT,
    openai_client=azure_client,
)

print("Azure model configured for the OpenAI Agents SDK.")

Azure model configured for the OpenAI Agents SDK.


Monthly Expenses collection Tool.

In [ ]:
def extract_pdf_text(file_path):
    reader = PdfReader(file_path)
    text = ""
    for page in reader.pages:
        page_text = page.extract_text()
        if page_text:
            text += page_text + "\n"
    return text


MONTHLY_EXPENSES_COLLECTION = {
    "monthly_expenses_pdf_1": {
        "filename": "MonthlySpendingData_Placeholder.pdf",
        "content": extract_pdf_text("placeholder/MonthlySpendingData_Placeholder.pdf")
    }
}


spending_data_text = MONTHLY_EXPENSES_COLLECTION["monthly_expenses_pdf_1"]["content"]
print("Loaded PDF text length:", len(spending_data_text), "characters")

Loaded PDF text length: 275 characters


In [ ]:
#parse_csv_transactions_tool()

In [ ]:
#calculate_spending_summary_tool()

In [ ]:
#save_transactions_tool()

Tool Prompt.

In [ ]:
instructions = f"""{RECOMMENDED_PROMPT_PREFIX}

You are Expense Analyst Agent, You are a Financial Advizing Agent that Analyzes financial data provided by users through 
CSV files, chat inputs, PDF files. You categorizes transactions, identifies spending patterns, and summarizes expenses for the user


You are to, when prompted by the user, to list the monthly expenses provided in the pdf from the MONTHLY_EXPENSES_COLLECTION and tell them how they could better optimize their spending.

. 

"""

ExpenseAnalystAgent = Agent(
    name="ExpenseAnalystAgent",
    handoff_description="Analyzes budget and summarizes spending.",
    instructions=instructions,
    model=azure_model,
)

print("Agent built:", ExpenseAnalystAgent.name)

#when executed with tools that can read the MONTHLY_EXPENSES_COLLECTION, the chatbot should be able to properly assess the file
#for now, its a pdf as a placeholder but we can change that later. 
# We will add more to the description of the agent we are creating the tools - Chris


Agent built: ExpenseAnalystAgent


CHAT WITH GRADIO!

In [ ]:
chat_session = SQLiteSession("gradio_user", "user_expenses.db")

async def chat(message, history):
    result = await Runner.run(ExpenseAnalystAgent, message, session=chat_session)
    return result.final_output

gr.ChatInterface(chat).launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7865


KeyboardInterrupt: 

In [ ]:
# Hey guys, so this is basically the skeleton of the Expenses Agent, I formatted it in a way that it will read a single pdf file when given the tools to do so
#We will end up adding memory as well, preferably after the tools are implemented for this agent
#It may be a little thrown together in terms of the structure, but I made what I could on my own. If you guys have issues with it, we will modify it together. 
#-Chris